In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from matplotlib import colors
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

from shaft_force_sensing.data import get_train_test, get_cols, SensorDataset

In [ ]:
ROOT = Path('..')
DATA_ROOT = ROOT / 'data'

In [ ]:
train_paths, test_paths = get_train_test(DATA_ROOT, '')

In [ ]:
groups = [
    'Free',
    'Palpation',
    'Traction'
]
groups = dict(zip(groups, [0] * len(groups)))

In [ ]:
for p in tqdm(train_paths):
    g = re.match(r'([a-zA-Z]+)_\d+', p.stem).groups()[0]
    df = pd.read_csv(p)
    groups[g] += len(df)

Train

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    length = v * 0.9
    if k == 'Free':
        length /= 4
    subset_lengths[k] = length

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Val

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    length = v * 0.1
    if k == 'Free':
        length /= 4
    subset_lengths[k] = length

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Test

In [ ]:
groups = [
    'Free',
    'Palpation',
    'Traction'
]
groups = dict(zip(groups, [0] * len(groups)))

In [ ]:
for p in tqdm(test_paths):
    g = re.match(r'([a-zA-Z]+)_\d+', p.stem).groups()[0]
    df = pd.read_csv(p)
    groups[g] += len(df)

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    subset_lengths[k] = v

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Distribution of the dataset

In [ ]:
hex10_forces = np.zeros((0, 3))
ati_forces = np.zeros((0, 3))

for p in tqdm(test_paths):
    df = pd.read_csv(p)
    hex10_forces = np.vstack((hex10_forces, df[['fx', 'fy', 'fz']].values))
    ati_forces = np.vstack((ati_forces, df[['ati_fx', 'ati_fy', 'ati_fz']].values))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase
from matplotlib.ticker import LogLocator, LogFormatterSciNotation, NullFormatter

# Single-column friendly size for double-column papers
fig, axes = plt.subplots(3, 1, figsize=(5, 3), sharex=True, dpi=300)
axis_names = ['x', 'y', 'z']
component_colors = {'x': '#CB3A27', 'y': '#6CAC52', 'z': '#517CB6'}
colors_map = {'ATI': '#5c5c5c'}
force_pairs = [
    ('ATI', ati_forces),
    ('HEX10', hex10_forces),
]

all_forces = np.concatenate([ati_forces.reshape(-1), hex10_forces.reshape(-1)])
if all_forces.size > 0:
    x_min = np.floor(all_forces.min() / 10.0) * 10.0
    x_max = np.ceil(all_forces.max() / 10.0) * 10.0
    if x_min == x_max:
        x_min -= 10.0
        x_max += 10.0
else:
    x_min, x_max = -10.0, 10.0

# Keep x ticks readable while forcing both extremes to appear.
span = x_max - x_min
tick_step = max(10.0, np.ceil(span / 50.0) * 10.0)
x_ticks = np.arange(x_min, x_max + 0.1, tick_step)
if x_ticks.size == 0 or x_ticks[-1] < x_max:
    x_ticks = np.append(x_ticks, x_max)
x_ticks = np.unique(x_ticks)

for axis_idx, axis_name in enumerate(axis_names):
    ax = axes[axis_idx]
    for label, force_values in force_pairs:
        color = component_colors[axis_name] if label == 'HEX10' else colors_map[label]
        ax.hist(
            force_values[:, axis_idx],
            bins=70,
            alpha=0.7 if label == 'HEX10' else 1.0,
            color=color,
            edgecolor='none',
            zorder=3 if label == 'HEX10' else 2,
        )
    ax.set_xlim(x_min, x_max + 0.5)
    ax.set_xticks(x_ticks)
    ax.text(
        0.04,
        0.60,
        f'$F_{axis_name}$',
        transform=ax.transAxes,
        fontsize=12,
        fontweight='bold',
        color=component_colors[axis_name],
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2.5),
    )
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=4))
    ax.yaxis.set_major_formatter(LogFormatterSciNotation(base=10, labelOnlyBase=True))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=8))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, which='major', linestyle='--', linewidth=0.8, color='#c4c4c4', alpha=0.45)
    ax.grid(False, which='minor')
    ax.tick_params(axis='both', labelsize=12, width=1.1, length=4, color='black')
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.1)

class HandlerTriColor(HandlerBase):
    def create_artists(
        self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans
    ):
        stripe_width = width / 3.0
        artists = []
        for i, c in enumerate(orig_handle):
            artists.append(
                Rectangle(
                    (xdescent + i * stripe_width, ydescent),
                    stripe_width,
                    height,
                    facecolor=c,
                    edgecolor='none',
                    transform=trans,
                )
            )
        return artists

axes[-1].set_xlabel('Force (N)', fontsize=12)
fig.supylabel('Frequency', fontsize=12, x=0.02)

legend_handles = [
    Patch(facecolor=colors_map['ATI'], edgecolor='none'),
    (component_colors['x'], component_colors['y'], component_colors['z']),
]
legend_labels = ['ATI', 'HEX10']

fig.legend(
    legend_handles,
    legend_labels,
    loc='upper center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
    fontsize=12,
    handler_map={tuple: HandlerTriColor()},
)

plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])
output_path = Path('../logs/results/hex10_vs_ati_hist.pdf')
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
for i in range(3):
    print(f"Axis {axis_names[i]}: ATI min={ati_forces[:, i].min():.2f}, "
          f"ATI max={ati_forces[:, i].max():.2f}, "
          f"HEX10 min={hex10_forces[:, i].min():.2f}, "
          f"HEX10 max={hex10_forces[:, i].max():.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde

axis_names = ['x', 'y', 'z']
axis_colors = {'x': '#b23a2a', 'y': '#2f7a53', 'z': '#2f3fa8'}

# Exponent on density to soften sharp peaks: <1 makes violins visually fuller.
density_exponent = 0.6
max_half_width = 0.42

def draw_half_violin(
    ax, data, center, side, facecolor, exponent=1.0, bw='scott', edgecolor='black', linecolor='black'
):
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if data.size < 2:
        return np.nan, None, None

    y_min, y_max = np.min(data), np.max(data)
    if y_min == y_max:
        y_min -= 1e-6
        y_max += 1e-6

    y = np.linspace(y_min, y_max, 256)
    kde = gaussian_kde(data, bw_method=bw)
    d = kde(y)
    d = d / d.max() if d.max() > 0 else d
    d = np.power(d, exponent)
    w = max_half_width * d

    x_edge = center - w if side == 'left' else center + w
    ax.fill_betweenx(
        y,
        center,
        x_edge,
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=0.9,
        alpha=0.78,
        zorder=2,
    )
    # Overlay KDE boundary line on top of each half violin.
    ax.plot(x_edge, y, color=linecolor, linewidth=1.1, alpha=0.95, zorder=3)
    return np.median(data), y, x_edge

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.8), dpi=300, sharey=False)

for i, axis_name in enumerate(axis_names):
    ax = axes[i]
    ati = ati_forces[:, i]
    hex10 = hex10_forces[:, i]

    ati_med, _, _ = draw_half_violin(
        ax,
        ati,
        center=1.0,
        side='left',
        facecolor='#8a8a8a',
        exponent=density_exponent,
        linecolor='#2f2f2f',
    )
    hex10_med, _, _ = draw_half_violin(
        ax,
        hex10,
        center=1.0,
        side='right',
        facecolor=axis_colors[axis_name],
        exponent=density_exponent,
        linecolor='#1a1a1a',
    )

    # Overlay box plots (one per source) to show quartiles and whiskers.
    bp_ati = ax.boxplot(
        [ati],
        positions=[0.93],
        widths=0.08,
        vert=True,
        patch_artist=True,
        showfliers=False,
        zorder=4,
    )
    bp_hex10 = ax.boxplot(
        [hex10],
        positions=[1.07],
        widths=0.08,
        vert=True,
        patch_artist=True,
        showfliers=False,
        zorder=4,
    )

    for box in bp_ati['boxes']:
        box.set(facecolor='white', edgecolor='#1f1f1f', linewidth=1.0)
    for med in bp_ati['medians']:
        med.set(color='#1f1f1f', linewidth=1.2)
    for w in bp_ati['whiskers'] + bp_ati['caps']:
        w.set(color='#1f1f1f', linewidth=1.0)

    for box in bp_hex10['boxes']:
        box.set(facecolor='white', edgecolor='#1f1f1f', linewidth=1.0)
    for med in bp_hex10['medians']:
        med.set(color='#1f1f1f', linewidth=1.2)
    for w in bp_hex10['whiskers'] + bp_hex10['caps']:
        w.set(color='#1f1f1f', linewidth=1.0)

    ax.scatter(1 - 0.06, ati_med, color='black', s=12, zorder=5)
    ax.scatter(1 + 0.06, hex10_med, color='white', edgecolors='black', s=12, zorder=5)

    ax.axvline(1.0, color='black', linewidth=0.8, alpha=0.3)
    ax.set_xlim(0.55, 1.45)
    ax.set_xticks([1.0])
    ax.set_xticklabels([f'F_{axis_name}'])
    ax.set_title(f'{axis_name.upper()} axis', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.35)

axes[0].set_ylabel('Force (N)')
fig.supxlabel(
    f'Half violin + KDE boundary + box plot (left=ATI, right=HEX10, exponent={density_exponent})',
    y=0.05,
)

legend_handles = [
    Patch(facecolor='#8a8a8a', edgecolor='black', label='ATI (left half)'),
    Patch(facecolor='#6a79d9', edgecolor='black', label='HEX10 (right half)'),
]
fig.legend(
    handles=legend_handles,
    loc='upper center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()